# RAG Pipeline with Caching



### Load the dataset

We will use a dataset specifically tailored for RAG tasks. The [RAG-Mini-Wikipedia](https://huggingface.co/datasets/rag-datasets/rag-mini-wikipedia) from Hugging Face is a curated, small-scale dataset designed to evaluate RAG pipelines. It provides ~900 questions, ground truth answers, and ~690,000 words of Wikipedia snippets for retrieving context.

Contains two main parts:
* *Passages*: A set of documents (passages) indexed for retrieval
* *QA Pairs*: ~900 questions with corresponding ground truth answers, often including difficulty ratings

* Core file: `data/passages.parquet` <br>
    This is the retrieval corpus. We will perform the semantic searches for regular queries here. <br> <br>
* Query/evaluation file: `data/test.parquet` <br>
    This is the evaluation dataset containing sample questions and their (ideal) answers. We can use various metrics to check the performance of the RAG system using this data.


Note that there are two data folders here. *`data`*, which is already structured for retrieval-style tasks, and *`raw_data`*, which is the pre-processing source. We don't need the raw texts for now. 

This is not raw text; it is pre-prepared passage-level data, so your pipeline can skip chunking. However, you can try using it to add a preprocessing stage (cleaning, normalisation, chunking etc.) to the RAG flow.

Let's read the parquet data files

In [1]:
import pandas as pd, numpy as np

passages = pd.read_parquet("rag-mini-wikipedia/data/passages.parquet")
passages.head()

# we do not need the 'test' file yet
# test = pd.read_parquet("rag-mini-wikipedia/data/test.parquet")

,passage
id,
0,"Uruguay (official full name in ; pron. , Eas..."
1,"It is bordered by Brazil to the north, by Arge..."
2,Montevideo was founded by the Spanish in the e...
3,The economy is largely based in agriculture (m...
4,"According to Transparency International, Urugu..."


In [2]:
passages.shape

(3200, 1)

### Prepare Documents

Since chunking is already done, the pipeline becomes

*User Query → Embedding → Vector Search (on passages.text) → Augment prompt → LLM generates answer*

Let's prepare documents for embedding

In [3]:
documents = passages["passage"].tolist()

In [4]:
docs = [
    {
        "text": row["passage"],
        "id": row.name
    }
    for _, row in passages.iterrows()
]

In [5]:
docs[0]

{'text': 'Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.',
 'id': 0}

### Adding Data to ChromaDB

#### Creating a Chroma Collection

Using a vectorDB like Chroma provides scalability, approximate nearest neighbour (ANN), and metadata filtering.

Let's embed and store the docs in a Chroma collection.

In [6]:
import chromadb

chroma = chromadb.Client()

In [7]:
store = chroma.get_or_create_collection("ChromaStore")

In [8]:
# recall that we need to provide the texts and metadatas separately
texts = [doc['text'] for doc in docs]
metadatas = [str(doc['id']) for doc in docs]

store.add(
    documents = texts,
    ids = metadatas,
    # embeddings = embeddings   # can directly provide embeddings
)

In [9]:
query = "Who won the election of 1860?"

chroma_results = store.query(query_texts = [query], n_results=3)

chroma_results

{'ids': [['202', '201', '319']],
 'embeddings': None,
 'documents': [['United States presidential election, 1856',
   'United States presidential election, 1848',
   'On November 6, 1860, Lincoln was elected as the 16th President of the United States, beating Democrat Stephen A. Douglas, John C. Breckinridge of the Southern Democrats, and John Bell of the new Constitutional Union Party. He was the first Republican president, winning entirely on the strength of his support in the North: he was not even on the ballot in nine states in the South, and won only 2 of 996 counties in the other Southern states. Lincoln gained 1,865,908 votes (39.9% of the total), for 180 electoral votes; Douglas, 1,380,202 (29.5%) for 12 electoral votes; Breckenridge, 848,019 (18.1%) for 72 electoral votes; and Bell, 590,901 (12.5%) for 39 electoral votes. There were fusion tickets in some states, but even if his opponents had combined in every state, Lincoln had a majority vote in all but two of the states in

### Augmentation and Generation

Now that we have the final top search results, we can pass it to an LLM along with the user query and a well-engineered prompt, to generate a direct answer to the query along with citations, rather than returning whole pages/chunks.

Initialise a client first

In [10]:
import os
from dotenv import load_dotenv
from google import genai

# Load API credentials
load_dotenv(override=True)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Initialise Client
client = genai.Client(api_key=GOOGLE_API_KEY)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Now, to pass the retrieved chunks to the LLM as context, let's join them

In [11]:
chroma_context = "\n\n".join(chroma_results['documents'][0])

In [12]:
def answer_query(query, context, llm = client):

    prompt = f"""Answer the question based only on the context below.
    Context: {context}
    
    Question:{query}"""

    response = client.models.generate_content(
        model = "gemini-2.5-flash-lite",
        contents = prompt
    )
    
    return response.text

In [13]:
chroma_response = answer_query(query, chroma_context)

In [14]:
chroma_response

'Lincoln won the election of 1860.'

## Caching

In a Retrieval-Augmented Generation (RAG) system, every query typically involves embedding the query, performing vector search, formatting retrieved context and generating a response using an LLM.

This pipeline is computationally expensive and introduces latency.

We can include a cache layer

## Caching in the RAG pipeline

In this last section we add different caching strategies on top of the basic RAG pipeline: response cache, retrieval cache, and a semantic cache.


### Where can we cache?

In a RAG system, we can cache at multiple stages of the pipeline:

* **Response cache**: store final answers for exactly repeated queries.
* **Retrieval cache**: store retrieved contexts for repeated queries.
* **Semantic cache**: reuse answers for *similar* queries using vector similarity instead of exact string match.


### Helper: retrieve context from Chroma

We first wrap the retrieval step into a helper function, so that we can reuse it across different cache types.


In [15]:
def get_context_from_chroma(query, n_results=3):
    """Run a vector search in the Chroma collection and return both context text and raw results."""
    results = store.query(query_texts=[query], n_results=n_results)
    context = "\n\n".join(results['documents'][0])
    return context, results


### 1. Response cache (exact match)

This is the simplest cache: if we have already answered the *same* query earlier, we return the stored answer instead of re-running retrieval and generation.


In [16]:
response_cache = {}

def answer_with_response_cache(query, n_results=3, llm=client):
    """Return an answer for the query using a simple exact-match response cache.

    The function returns a tuple `(answer, from_cache)` where `from_cache` is a boolean flag."""
    key = query.strip().lower()

    # 1. Check cache first
    if key in response_cache:
        return response_cache[key], True

    # 2. Otherwise, run the normal RAG pipeline
    context, _ = get_context_from_chroma(query, n_results=n_results)
    answer = answer_query(query, context, llm=llm)

    # 3. Store in cache for future calls
    response_cache[key] = answer

    return answer, False


In [17]:
query = "Who won the election of 1860?"

first_answer, first_from_cache = answer_with_response_cache(query)
second_answer, second_from_cache = answer_with_response_cache(query)

first_from_cache, second_from_cache


(False, True)

### 2. Retrieval cache

Instead of caching only the final answer, we can cache the *retrieved context* for a query. This is useful if we want to reuse the same retrieved passages with slightly different prompts, or run multiple LLM calls on the same context (for example: answer + explanation + follow-up).


In [18]:
retrieval_cache = {}

def get_context_with_retrieval_cache(query, n_results=3):
    """Return `(context, results)` and a `from_cache` flag for the given query."""
    key = (query.strip().lower(), n_results)

    # 1. Check if we have cached retrieval for this key
    if key in retrieval_cache:
        return retrieval_cache[key], True

    # 2. Otherwise, run retrieval and cache the results
    context, results = get_context_from_chroma(query, n_results=n_results)
    retrieval_cache[key] = (context, results)

    return (context, results), False


In [19]:
(context_1, results_1), from_cache_1 = get_context_with_retrieval_cache("Who won the election of 1860?")
(context_2, results_2), from_cache_2 = get_context_with_retrieval_cache("Who won the election of 1860?")

from_cache_1, from_cache_2


(False, True)

### 3. Semantic cache

A **semantic cache** lets us reuse answers for *similar* queries, even when the strings are not identical. We use another Chroma collection to store past questions and answers, and then perform a vector search over that cache.


In [20]:
semantic_cache = chroma.get_or_create_collection("SemanticResponseCache")



In [21]:
def answer_with_semantic_cache(query, threshold=0.25, n_results=3, llm=client):
    """Return an answer for the query using a semantic cache backed by Chroma.

    The function returns `(answer, from_cache, distance, metadata)`."""

    # 1. Try to find a similar question in the cache
    if semantic_cache.count() > 0:
        cached = semantic_cache.query(
            query_texts=[query],
            n_results=1,
            include=["documents", "metadatas", "distances"]
        )
        distance = cached['distances'][0][0]
        if distance < threshold:
            answer = cached['documents'][0][0]
            metadata = cached['metadatas'][0][0]
            return answer, True, distance, metadata

    # 2. No good semantic match found: run normal RAG
    context, results = get_context_from_chroma(query, n_results=n_results)
    answer = answer_query(query, context, llm=llm)

    # 3. Add this Q&A pair to the semantic cache
    semantic_cache.add(
        documents=[answer],
        metadatas=[{"question": query, "context": context}],
        ids=[str(semantic_cache.count())]
    )

    return answer, False, None, {"question": query, "context": context}


In [22]:
q1 = "Who won the election of 1860?"
q2 = "Who was elected US president in 1860?"

ans1, cache1, dist1, meta1 = answer_with_semantic_cache(q1)
ans2, cache2, dist2, meta2 = answer_with_semantic_cache(q2)

cache1, cache2, dist2


(False, False, None)